# Pinecone VectorStore

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- Pinecone의 완전 관리형 클라우드 벡터 DB 특징을 설명할 수 있어요
- Pinecone 인덱스를 생성하고 문서를 업로드할 수 있어요
- 로컬(Chroma/FAISS) vs 클라우드(Pinecone)의 차이를 이해하고 선택할 수 있어요

## 사전 지식

- 노트북 01(Chroma), 02(FAISS) 학습 완료
- Pinecone API 키 (없으면 해당 섹션을 건너뛰고 비교표를 확인해요)

---

## 로컬 vs 클라우드 VectorStore

```mermaid
flowchart LR
    subgraph Local["로컬 (Chroma / FAISS)"]
        LA["내 서버에 저장"]:::storage
        LB["인프라 직접 관리"]:::process
        LC["무료 오픈소스"]:::output
    end

    subgraph Cloud["클라우드 (Pinecone)"]
        CA["Pinecone 서버에 저장"]:::external
        CB["완전 관리형<br/>(인프라 불필요)"]:::process
        CC["무제한 확장<br/>Free Tier 제공"]:::output
    end

    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

> 🎯 **강의 포인트**: 로컬 VectorStore는 **개발·테스트** 단계에 적합하고, Pinecone 같은 클라우드 VectorStore는 **운영 환경**에 적합해요. 실제 서비스에서는 장애 복구, 백업, 스케일아웃을 직접 구현하기 어렵기 때문에 관리형 서비스가 유리해요.

**Pinecone을 선택하는 경우**:
- 운영 환경에서 안정적인 서비스를 운영해야 할 때
- 수백만 개 이상의 벡터를 다뤄야 할 때
- 서버 관리 부담 없이 벡터 DB를 사용하고 싶을 때

**API 키 발급**: [Pinecone Console](https://app.pinecone.io/)에서 무료로 발급받을 수 있어요. Free Tier는 월 100만 쿼리까지 무료예요.

**참고 문서**: [Pinecone 공식 문서](https://docs.pinecone.io/) | [LangChain Pinecone 통합](https://python.langchain.com/docs/integrations/vectorstores/pinecone/)

## API 키 설정

1. [Pinecone Console](https://app.pinecone.io/)에서 API 키 발급
2. `.env` 파일에 추가: `PINECONE_API_KEY="your-key"`

> **주의**: API 키가 없으면 인덱스 생성과 VectorStore 생성 셀은 건너뛰고 마무리 비교표를 확인해요.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    print("⚠️ PINECONE_API_KEY가 없습니다. .env 파일에 추가해주세요.")
else:
    print("✅ API 키가 로드되었습니다.")


✅ API 키가 로드되었습니다.


---

## 데이터 준비

In [2]:

# ---------------------------------------------------
# 문서 로드 및 분할 (Pinecone에 업로드할 데이터 준비)
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
loader = TextLoader("data/nlp-keywords.txt")
docs = loader.load_and_split(text_splitter)

print(f"문서 개수: {len(docs)}개")


문서 개수: 15개


---

## Pinecone 인덱스 생성 (최초 1회)

Pinecone 인덱스는 벡터 DB의 "테이블" 같은 개념이에요. 한 번 만들면 재사용할 수 있어요.

**중요 설정**:
- `dimension=1536`: OpenAI `text-embedding-3-small` 모델의 벡터 차원
- `metric="cosine"`: 코사인 유사도로 검색
- `ServerlessSpec`: 서버리스 방식 (사용량 기반 과금, 가장 경제적)

> ⚠️ **자주 하는 실수**: `dimension`은 사용하는 **임베딩 모델 차원과 반드시 일치**해야 해요. `text-embedding-3-small`은 1536, `text-embedding-3-large`는 3072차원이에요. 차원이 다르면 업로드 시 오류가 발생해요.

> 💡 **실무 팁**: 인덱스 생성 후 차원은 변경할 수 없어요. 임베딩 모델을 바꾸려면 새 인덱스를 만들고 모든 문서를 다시 업로드해야 해요. 처음부터 모델 선택을 신중하게 하세요.

In [3]:
# ---------------------------------------------------
# Pinecone 서버리스 인덱스 생성 (최초 1회)
# ---------------------------------------------------

# ============================================================
# TODO: Pinecone 인덱스를 생성해보세요 (없으면 건너뜁니다)
# 힌트: pc.create_index(name, dimension=1536, metric="cosine", spec=ServerlessSpec(...))
# 예상 결과: "인덱스 생성됨" 또는 "존재함" 출력
# ============================================================

from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "langchain-demo"

if not PINECONE_API_KEY:
    print("⚠️ PINECONE_API_KEY가 없어 인덱스 생성 단계를 건너뜁니다.")
else:
    # 1단계: Pinecone 클라이언트 초기화
    pc = Pinecone(api_key=PINECONE_API_KEY)

    # 2단계: 기존 인덱스 확인
    existing = [idx.name for idx in pc.list_indexes()]

    # 3단계: 없으면 새로 생성
    if INDEX_NAME not in existing:
        # dimension=1536: OpenAI text-embedding-3-small의 벡터 차원
        # metric="cosine": 코사인 유사도로 검색
        # ServerlessSpec: 서버리스 방식 (사용량 기반 과금)
        pc.create_index(
            name=INDEX_NAME,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print(f"✅ 인덱스 '{INDEX_NAME}' 생성됨")
    else:
        print(f"✅ 인덱스 '{INDEX_NAME}' 존재함")


✅ 인덱스 'langchain-demo' 존재함


---

## VectorStore 생성 및 문서 추가

`from_documents()`로 문서를 임베딩해서 Pinecone 클라우드에 업로드해요. Chroma/FAISS와 사용법이 거의 같아요.

> 🎯 **강의 포인트**: `from_documents()`를 Pinecone에 사용하면 내부적으로 문서를 배치(batch)로 나눠서 업로드해요. 수만 개의 문서를 한꺼번에 전송하는 대신 적절한 크기로 나눠 안정적으로 처리해요. 대용량 문서 업로드 시 네트워크 타임아웃 문제를 방지하는 방식이에요.

In [4]:
# ---------------------------------------------------
# Pinecone VectorStore 생성 및 문서 업로드
# ---------------------------------------------------

# ============================================================
# TODO: PineconeVectorStore.from_documents()로 문서를 클라우드에 업로드해보세요
# 힌트: PineconeVectorStore.from_documents(documents, embedding, index_name=...)
# 예상 결과: "문서가 Pinecone에 업로드되었습니다." 출력
# ============================================================

from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

if not PINECONE_API_KEY:
    vectorstore = None
    print("⚠️ API 키가 없어 Pinecone VectorStore 예제를 건너뜁니다.")
else:
    # 1단계: VectorStore 생성 및 문서 업로드
    # from_documents(): 임베딩 후 Pinecone 클라우드에 자동 업로드
    vectorstore = PineconeVectorStore.from_documents(
        documents=docs[:10],  # 샘플로 10개만 사용
        embedding=OpenAIEmbeddings(),
        index_name=INDEX_NAME
    )

    print("✅ 문서가 Pinecone에 업로드되었습니다.")


✅ 문서가 Pinecone에 업로드되었습니다.


---

## 유사도 검색

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


# ---------------------------------------------------
# Pinecone 유사도 검색 수행
# ---------------------------------------------------

# similarity_search(): Chroma/FAISS와 동일한 API — 클라우드 검색이지만 사용법은 동일
if not PINECONE_API_KEY or vectorstore is None:
    print("⚠️ VectorStore가 없어 검색을 수행할 수 없습니다.")
else:
    results = vectorstore.similarity_search("임베딩과 벡터 표현", k=3)

    print("=== 검색 결과 ===")
    for idx, doc in enumerate(results, 1):
        print(f"\n[{idx}] {doc.page_content[:120]}...")


=== 검색 결과 ===

[1] Crawling

정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하...

[2] Crawling

정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하...

[3] Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성"...


---

## 메타데이터 필터링

Pinecone도 메타데이터 필터를 지원해요. Chroma와 비슷하게 `filter` 파라미터를 사용해요.

In [6]:

# ---------------------------------------------------
# 메타데이터 필터링으로 특정 출처 문서만 검색
# ---------------------------------------------------

# filter: Pinecone도 Chroma와 동일한 방식으로 메타데이터 조건 지정 가능
if not PINECONE_API_KEY or vectorstore is None:
    print("⚠️ VectorStore가 없어 필터 검색을 수행할 수 없습니다.")
else:
    filtered_results = vectorstore.similarity_search(
        "자연어 처리",
        k=2,
        filter={"source": "data/nlp-keywords.txt"}
    )

    print("[필터링 결과]")
    for idx, doc in enumerate(filtered_results, 1):
        print(f"{idx}. {doc.page_content[:100]}...")


[필터링 결과]
1. HuggingFace

정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 ...
2. HuggingFace

정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 ...


---

## 마무리

### VectorStore 종합 비교

| 특징 | Chroma | FAISS | Pinecone |
|------|--------|-------|----------|
| 저장 위치 | 로컬 디스크 | 메모리 | 클라우드 |
| 검색 속도 | 중간 | 빠름 | 빠름 |
| 적합한 규모 | 소~중 | 중~대 | 무제한 |
| 비용 | 무료 | 무료 | Free Tier 있음 |
| 인프라 관리 | 직접 | 직접 | 관리형 |
| 추천 용도 | 개발·소규모 | 대규모·고속 | 운영 환경 |

### 선택 가이드

- **개발·프로토타입**: Chroma (설치가 가장 간단해요)
- **대규모 로컬 처리**: FAISS (빠르고 무료예요)
- **운영 환경**: Pinecone (관리 부담 없이 안정적이에요)

### 다음 학습

**6-4 노트북 04**: Milvus — 자체 서버에서 운영하는 대규모 오픈소스 벡터 DB를 배워볼게요.